# 42-1 Preprocesamiento de Texto con PySpark

Este notebook demuestra técnicas comunes de preprocesamiento de texto usando funciones nativas de PySpark, sin dependencias externas de NLP.

## Objetivo
Aplicar un pipeline de limpieza de texto que incluye: expansión de contracciones, normalización (minúsculas + eliminación de puntuación), remoción de acentos y eliminación de stopwords.

## Flujo del notebook
1. Instalación de dependencias.
2. Importación de módulos PySpark y utilidades de texto.
3. Creación de la sesión Spark.
4. Creación de datos de ejemplo (oraciones en inglés y español).
5. Expansión de contracciones (`contractions`).
6. Normalización: minúsculas y eliminación de puntuación con PySpark nativo.
7. Remoción de acentos (`unidecode`).
8. Tokenización y eliminación de stopwords con `StopWordsRemover` de PySpark ML.

## Antes de ejecutar
- Verifica que la sesión de Spark esté activa.
- Ejecuta las celdas en orden para mantener dependencias.


## 1. Instalación de dependencias
Se instalan librerías para visualización, manejo de datos, lectura de PDFs, expansión de contracciones y remoción de acentos. No se requiere ninguna librería externa de NLP.


In [ ]:
%pip install numpy pandas matplotlib seaborn wordcloud pypdf2 contractions unidecode

## 2. Importación de módulos
Se importan los módulos de PySpark (sesión, tipos, funciones), `PyPDF2`, `os` y `Pipeline` de PySpark ML.


In [ ]:
import findspark
findspark.init('/Users/joseaguilar/Documents/Development/spark/spark-3.5.1-bin-hadoop3')
from pyspark import *
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, DateType, TimestampType, LongType
from pyspark.sql.types import ArrayType, DoubleType, BooleanType, DecimalType
from pyspark.sql.functions import regexp_extract, split, from_unixtime, col, avg, min, max
from pyspark.sql.functions import grouping_id, window, explode, to_json, from_json
from pyspark.sql.functions import udf, lit, current_timestamp, current_date, date_format
from pyspark.ml import PipelineModel, Pipeline
import PyPDF2
import os

## 3. Creación de la sesión Spark
Se inicializa una `SparkSession` básica sin dependencias externas de NLP.


In [ ]:
spark = SparkSession.builder \
    .appName("preprocesamiento_texto") \
    .getOrCreate()

## 4. Importaciones adicionales para procesamiento de texto
Se importan funciones nativas de PySpark para normalización (`lower`, `regexp_replace`, `trim`, `split`) y la versión de Spark/Python.


In [ ]:
from pyspark.sql.functions import lower, regexp_replace, trim, split, explode, col, size

print(f"Spark version: {spark.version}")
import sys
print(f"Python version: {sys.version}")

## 5. Creación de datos de ejemplo
Se crean oraciones de ejemplo en inglés (con contracciones) y en español para demostrar las técnicas de preprocesamiento.


In [ ]:
oraciones = [
    "It's essential to drink coffee at a café and to keep one’s emotions in check while analyzing complex, real-time data!",
    "I'm going to the store because I can't wait to buy some groceries.",
    "El número pi es aproximadamente 3,1416. Ya ahora saben."
]
df_oraciones = spark.createDataFrame([(s,) for s in oraciones], ["texto"])
df_oraciones.show(truncate=False)

## 6. Expansión de contracciones
Se utiliza la librería `contractions` mediante una UDF para expandir contracciones en inglés (e.g., "I'm" → "I am", "can't" → "cannot").


In [ ]:
import contractions
from pyspark.sql.functions import col, udf

# Definir una UDF para expandir contracciones en inglés
expand_contr = udf(lambda text: contractions.fix(text))
df_expanded = df_oraciones.withColumn("texto_exp", expand_contr(col("texto")))
df_expanded.select("texto", "texto_exp").show(truncate=False)


## 7. Normalización: minúsculas y eliminación de puntuación
Se usan funciones nativas de PySpark para convertir a minúsculas (`lower`), eliminar puntuación (`regexp_replace`) y limpiar espacios (`trim`). El resultado es una columna `texto_limpio` con una lista de tokens normalizados.


In [ ]:
# Normalización usando funciones nativas de PySpark
# 1. Convertir a minúsculas
df_clean = df_expanded.withColumn("texto_norm", lower(col("texto_exp")))

# 2. Eliminar puntuación y caracteres no deseados (conservar letras, dígitos, acentos y espacios)
df_clean = df_clean.withColumn(
    "texto_norm",
    regexp_replace(col("texto_norm"), r"[^\w\sáéíóúÁÉÍÓÚüÜñÑ]", "")
)

# 3. Reemplazar múltiples espacios por uno solo y hacer trim
df_clean = df_clean.withColumn(
    "texto_norm",
    trim(regexp_replace(col("texto_norm"), r"\s+", " "))
)

# 4. Tokenizar (dividir en palabras)
df_clean = df_clean.withColumn("texto_limpio", split(col("texto_norm"), " "))

df_clean.select("texto_exp", "texto_limpio").show(truncate=False)

## 8. Remoción de acentos
Se utiliza la librería `unidecode` mediante una UDF para eliminar acentos y caracteres especiales del texto (e.g., "café" → "cafe", "número" → "numero").


In [ ]:
import unidecode
remove_accents = udf(lambda text: unidecode.unidecode(text))
df_no_accents = df_clean.withColumn("texto_sin_acentos", remove_accents(col("texto_exp")))
df_no_accents.select("texto_exp", "texto_sin_acentos").show(truncate=False)


## 9. Tokenización y eliminación de stopwords
Se usa `StopWordsRemover` de PySpark ML para eliminar palabras vacías (stopwords) en inglés y español. Se tokeniza el texto sin acentos, se aplica el filtro y se obtiene la columna `tokens_final`.


In [ ]:
from pyspark.ml.feature import StopWordsRemover

# 1. Tokenizar el texto sin acentos (dividir en palabras)
df_tokenized = df_no_accents.withColumn(
    "tokens",
    split(
        trim(regexp_replace(lower(col("texto_sin_acentos")), r"[^\w\sáéíóúüñ]", "")),
        r"\s+"
    )
)

# 2. Definir stopwords personalizadas (combinación EN/ES)
custom_stopwords = ["y", "de", "la", "el", "the", "and", "is", "es", "porque",
                    "to", "am", "in", "some", "a", "an", "at", "on", "for",
                    "que", "en", "un", "una", "los", "las", "del", "al"]

# 3. Aplicar StopWordsRemover de PySpark ML
stopwords_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="tokens_final",
    stopWords=custom_stopwords,
    caseSensitive=False
)

df_tokens_final = stopwords_remover.transform(df_tokenized)
df_tokens_final.select("texto_limpio", "tokens_final").show(truncate=False)